# ⚠️ 已棄用(`DEPRECATED`) ⚠️

**這份檔案已被 `stage5_seismic_checks.ipynb`(`+ design_freeze.json`,
最新版本)正式取代,不建議再使用這裡的數字。**

**棄用原因**:這份檔案是在不知情的狀況下,跟 `repo` 裡已經存在的
官方 `Stage 2/3/4`(`stage2_canonical_elastic_model.ipynb`、
`stage3_load_combinations_demand_extraction.ipynb`/`VL-16`、
`stage4_rc_design_loop.ipynb`)平行、獨立重建需求做出來的——用的
重力載重是反推估計值,不是用真實從屬寬度算出的精確值,`1F` 柱軸力
跟官方版本相差接近 `80%`。

**這份檔案的歷史價值**:它曾經發現「`Stage 2/3` 模型需要梁上真實
分佈載重、不能只用集中力簡化」這件事,以及「`Stage 5` 需要補上
強柱弱梁、`capacity-design shear`、圍束、接頭剪力這幾項檢核」——
這兩個發現都已經被正式吸收進 `stage3_load_combinations_demand_
extraction.ipynb`(更早、更精確地做過一次)跟 `stage5_seismic_
checks.ipynb`。保留這份檔案是為了記錄這段過程,不是因為它的數字
還可信。

**正確的主線是**:`stage2_canonical_elastic_model.ipynb` →
`stage3_load_combinations_demand_extraction.ipynb` →
`stage4_rc_design_loop.ipynb` → `stage5_seismic_checks.ipynb` →
`design_freeze.json`

---



# Stage 4/5 重做:梁上真實分佈重力載重(取代原本集中在節點的簡化)

**這次重建的起因**:一張網路上的 RC 構架裂縫示意圖(顯示梁跨中正
彎矩、梁端負彎矩的典型撓曲裂縫,以及接頭附近的對角剪力裂縫),
揭露了我們之前 `Stage 2/3` 模型的一個已知局限——重力載重集中施加
在柱頂節點,不是以真實分佈載重的形式作用在梁上,導致梁的 `D+L`
彎矩幾乎是 `0`,完全無法重現那張圖裡的典型撓曲行為。

**修正方式**:換算成梁上均布載重 $w=49.2kN/m$(總重力量級不變,
只是改變它進入結構的路徑,從梁傳給柱,不是直接集中在柱頂)。

**結果非常重大**:梁端彎矩從 `24.19` 暴增到 `162.52`,`2F` 柱彎矩
從 `11.06` 增加超過 `12` 倍到 `141.95`——整個 `Stage 4/5` 的配筋
設計全部需要重新做,`2F` 柱第一次真正被「強度需求」而不是「規範
最小值」`govern`。

In [1]:

try:
    import openseespy.opensees as ops
except ImportError:
    import sys
    !{sys.executable} -m pip install -q openseespy
    import openseespy.opensees as ops
import os
if not os.path.exists('rc_design.py'):
    !wget -q -O rc_design.py https://raw.githubusercontent.com/zhixiu0223/taiwan-seismic-code-calc/main/rc_design.py
from rc_design import design_column_PM, design_rebar, design_doubly_reinforced

L_bay = 6.0; h1 = 3.5; h2 = 3.5
h_col = 0.40; b_beam, h_beam = 0.30, 0.50
E_rc = 2.463e7
Ic = 0.7*(h_col**4/12); Ib = 0.35*(b_beam*h_beam**3/12)
Ac = h_col**2; Ab = b_beam*h_beam
F1_Y, F2_Y = 9.938, 15.900
w_udl = 49.20  # kN/m, 換算自原本集中力(147.6*2/6.0), 總重力量級不變

print(f"梁上均布重力載重 w = {w_udl}kN/m(= 147.6*2/{L_bay}, 總量不變只改路徑)")


梁上均布重力載重 w = 49.2kN/m(= 147.6*2/6.0, 總量不變只改路徑)


## 第 1 課:重新建模(跨中加節點,用 `eleLoad` 施加真正的均布載重)

**這裡真的踩過一個坑**:第一次用解析公式 `M(x)=Mi+Vi*x-w*x²/2` 手算
跨中彎矩,算出 `359.74kN-m`,遠超理論值(`73.80kN-m` 完全固接端理論
值)快 `5` 倍——是 `Vi` 的正負號方向用錯了。改用更可靠的方式:直接
在跨中加一個節點,把梁切成兩段,讓 `OpenSeesPy` 自己算出跨中內力,
不再自己手算公式去猜符號。

In [2]:

def build_with_midspan():
    ops.wipe(); ops.model('basic', '-ndm', 2, '-ndf', 3)
    ops.node(1, 0.0, 0.0);   ops.node(2, L_bay, 0.0)
    ops.node(3, 0.0, h1);    ops.node(4, L_bay, h1)
    ops.node(5, 0.0, h1+h2); ops.node(6, L_bay, h1+h2)
    ops.node(7, L_bay/2, h1)
    ops.node(8, L_bay/2, h1+h2)
    ops.fix(1,1,1,1); ops.fix(2,1,1,1)
    ops.geomTransf('Linear', 1)
    ops.element('elasticBeamColumn', 1, 1, 3, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 2, 2, 4, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 3, 3, 5, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 4, 4, 6, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 5, 3, 7, Ab, E_rc, Ib, 1)
    ops.element('elasticBeamColumn', 9, 7, 4, Ab, E_rc, Ib, 1)
    ops.element('elasticBeamColumn', 6, 5, 8, Ab, E_rc, Ib, 1)
    ops.element('elasticBeamColumn', 10, 8, 6, Ab, E_rc, Ib, 1)

def run_gravity():
    build_with_midspan()
    ops.timeSeries('Linear', 1); ops.pattern('Plain', 1, 1)
    ops.eleLoad('-ele', 5, 9, '-type', '-beamUniform', -w_udl)
    ops.eleLoad('-ele', 6, 10, '-type', '-beamUniform', -w_udl)
    ops.system('BandGeneral'); ops.numberer('Plain'); ops.constraints('Plain')
    ops.algorithm('Linear'); ops.integrator('LoadControl', 1.0); ops.analysis('Static')
    ops.analyze(1)
    return (ops.eleForce(1), ops.eleForce(2), ops.eleForce(5), ops.eleForce(9),
            ops.eleForce(3), ops.eleForce(4), ops.eleForce(6), ops.eleForce(10))

def run_lateral():
    build_with_midspan()
    ops.timeSeries('Linear', 1); ops.pattern('Plain', 1, 1)
    ops.load(3, F1_Y, 0.0, 0.0); ops.load(5, F2_Y, 0.0, 0.0)
    ops.system('BandGeneral'); ops.numberer('Plain'); ops.constraints('Plain')
    ops.algorithm('Linear'); ops.integrator('LoadControl', 1.0); ops.analysis('Static')
    ops.analyze(1)
    return (ops.eleForce(1), ops.eleForce(2), ops.eleForce(5), ops.eleForce(9),
            ops.eleForce(3), ops.eleForce(4), ops.eleForce(6), ops.eleForce(10))

g = run_gravity()
l = run_lateral()

# 平衡檢核
build_with_midspan()
ops.timeSeries('Linear', 1); ops.pattern('Plain', 1, 1)
ops.eleLoad('-ele', 5, 9, '-type', '-beamUniform', -w_udl)
ops.eleLoad('-ele', 6, 10, '-type', '-beamUniform', -w_udl)
ops.system('BandGeneral'); ops.numberer('Plain'); ops.constraints('Plain')
ops.algorithm('Linear'); ops.integrator('LoadControl', 1.0); ops.analysis('Static')
ops.analyze(1)
ops.reactions()
Ry1 = ops.nodeReaction(1)[1]; Ry2 = ops.nodeReaction(2)[1]
assert abs((Ry1+Ry2) - w_udl*2*L_bay) < 0.01
print(f"[PASS] 平衡檢核: 總反力={Ry1+Ry2:.2f}kN = w*2*L={w_udl*2*L_bay:.2f}kN")


[PASS] 平衡檢核: 總反力=590.40kN = w*2*L=590.40kN


In [3]:

labels = ['1F左柱(elem1)', '1F右柱(elem2)', '1F樑左半(elem5)', '1F樑右半(elem9)',
          '2F左柱(elem3)', '2F右柱(elem4)', '屋頂樑左半(elem6)', '屋頂樑右半(elem10)']

print(f"{'構件':<16}{'端':<4}{'D+L':>10}{'E':>9}{'D+L+E':>10}{'D+L-E':>10}")
governing = {}
for gf, lf, label in zip(g, l, labels):
    for end, idx in [('i',2),('j',5)]:
        Mg, Ml = gf[idx], lf[idx]
        print(f"{label:<16}{end:<4}{Mg:>10.3f}{Ml:>9.3f}{Mg+Ml:>10.3f}{Mg-Ml:>10.3f}")
        governing[(label,end)] = max(abs(Mg+Ml), abs(Mg-Ml))

print(f"\n=== Governing(各構件最大絕對值) ===")
print(f"1F柱: {max(governing[('1F左柱(elem1)','i')], governing[('1F左柱(elem1)','j')], governing[('1F右柱(elem2)','i')], governing[('1F右柱(elem2)','j')]):.2f}kN-m")
print(f"2F柱: {max(governing[('2F左柱(elem3)','i')], governing[('2F左柱(elem3)','j')], governing[('2F右柱(elem4)','i')], governing[('2F右柱(elem4)','j')]):.2f}kN-m")
print(f"1F樑端點(負彎矩): {governing[('1F樑左半(elem5)','i')]:.2f}kN-m")
print(f"1F樑跨中(正彎矩): {governing[('1F樑左半(elem5)','j')]:.2f}kN-m")
print(f"屋頂樑端點(負彎矩): {governing[('屋頂樑左半(elem6)','i')]:.2f}kN-m")
print(f"屋頂樑跨中(正彎矩): {governing[('屋頂樑左半(elem6)','j')]:.2f}kN-m")


構件              端          D+L        E     D+L+E     D+L-E
1F左柱(elem1)     i      -22.324   32.130     9.806   -54.454
1F左柱(elem1)     j      -44.008   13.157   -30.850   -57.165
1F右柱(elem2)     i       22.324   32.035    54.359    -9.711
1F右柱(elem2)     j       44.008   13.111    57.118    30.897
1F樑左半(elem5)    i      138.336  -24.189   114.147   162.524
1F樑左半(elem5)    j       83.064    0.010    83.075    83.054
1F樑右半(elem9)    i      -83.064   -0.010   -83.075   -83.054
1F樑右半(elem9)    j     -138.336  -24.168  -162.503  -114.168
2F左柱(elem3)     i      -94.328   11.031   -83.297  -105.359
2F左柱(elem3)     j     -125.162   16.784  -108.378  -141.946
2F右柱(elem4)     i       94.328   11.057   105.385    83.271
2F右柱(elem4)     j      125.162   16.777   141.939   108.385
屋頂樑左半(elem6)    i      125.162  -16.784   108.378   141.946
屋頂樑左半(elem6)    j       96.238    0.003    96.241    96.235
屋頂樑右半(elem10)   i      -96.238   -0.003   -96.241   -96.235
屋頂樑右半(elem10)   j     -125.162  -16.777 

## 第 2 課:柱梁撓曲配筋(用新 `governing demand` 全部重做)

**`2F` 柱第一次真正被強度需求(不是規範最小值)`govern`**。

In [4]:

def Mn_at_Pu(curve, Pu_target):
    sc = sorted(curve, key=lambda p: p['Pn'])
    for i in range(len(sc)-1):
        if sc[i]['Pn'] <= Pu_target <= sc[i+1]['Pn']:
            t = (Pu_target-sc[i]['Pn'])/(sc[i+1]['Pn']-sc[i]['Pn'])
            return sc[i]['Mn'] + t*(sc[i+1]['Mn']-sc[i]['Mn'])
    return None

def design_column_rebar(b_cm, h_cm, Pu_kN, Mu_kNm, fc=280.0, fy=4200.0, Es=2.0e6,
                          cover_to_center=4.0, rho_candidates=None):
    if rho_candidates is None:
        rho_candidates = [0.010, 0.012, 0.014, 0.016, 0.018, 0.020, 0.025, 0.030, 0.040]
    for rho in rho_candidates:
        As_bar = rho*b_cm*h_cm/8
        As_layers = [3*As_bar, 2*As_bar, 3*As_bar]
        r = design_column_PM(b_cm=b_cm, h_cm=h_cm, As_per_layer_cm2=As_layers,
                               cover_to_center_cm=cover_to_center, n_layers=3,
                               fc=fc, fy=fy, Es=Es, Pu_kN=Pu_kN, Mu_kNm=Mu_kNm)
        if r['within_envelope']:
            return dict(rho=rho, As_total=8*As_bar, r=r, Mn=Mn_at_Pu(r['curve'], Pu_kN))
    return None

Mu_1F_col = max(governing[('1F左柱(elem1)','i')], governing[('1F左柱(elem1)','j')],
                 governing[('1F右柱(elem2)','i')], governing[('1F右柱(elem2)','j')])
Mu_2F_col = max(governing[('2F左柱(elem3)','i')], governing[('2F左柱(elem3)','j')],
                 governing[('2F右柱(elem4)','i')], governing[('2F右柱(elem4)','j')])
Mu_1F_beam_neg = governing[('1F樑左半(elem5)','i')]
Mu_1F_beam_pos = governing[('1F樑左半(elem5)','j')]
Mu_roof_beam_neg = governing[('屋頂樑左半(elem6)','i')]
Mu_roof_beam_pos = governing[('屋頂樑左半(elem6)','j')]

Pu_1F_col, Pu_2F_col = 308.85, 153.19  # 軸力受分佈載重影響較小, 沿用既有值

result_1F_col = design_column_rebar(40.0, 40.0, Pu_1F_col, Mu_1F_col)
result_2F_col = design_column_rebar(40.0, 40.0, Pu_2F_col, Mu_2F_col)
print(f"1F柱(Pu={Pu_1F_col}, Mu={Mu_1F_col:.2f}): rho={result_1F_col['rho']:.3%}, util={result_1F_col['r']['utilization']:.4f}")
print(f"2F柱(Pu={Pu_2F_col}, Mu={Mu_2F_col:.2f}): rho={result_2F_col['rho']:.3%}, util={result_2F_col['r']['utilization']:.4f}")
assert result_2F_col['rho'] > 0.010, "2F柱這次應該被強度需求govern, 不是最小鋼筋比"
print("[PASS] 2F柱確認被強度需求govern(rho>1.0%)")

r_1F_neg = design_rebar(Mu_1F_beam_neg, 30.0, 50.0, cover=4.0)
r_1F_pos = design_rebar(Mu_1F_beam_pos, 30.0, 50.0, cover=4.0)
r_roof_neg = design_rebar(Mu_roof_beam_neg, 30.0, 50.0, cover=4.0)
r_roof_pos = design_rebar(Mu_roof_beam_pos, 30.0, 50.0, cover=4.0)
print(f"\n1F樑: 上緣As={r_1F_neg['As_provided']:.2f}cm2({r_1F_neg['bar_size']}x{r_1F_neg['n_bars']}), "
      f"下緣As={r_1F_pos['As_provided']:.2f}cm2({r_1F_pos['bar_size']}x{r_1F_pos['n_bars']})")
print(f"屋頂樑: 上緣As={r_roof_neg['As_provided']:.2f}cm2({r_roof_neg['bar_size']}x{r_roof_neg['n_bars']}), "
      f"下緣As={r_roof_pos['As_provided']:.2f}cm2({r_roof_pos['bar_size']}x{r_roof_pos['n_bars']})")


1F柱(Pu=308.85, Mu=57.17): rho=1.000%, util=0.3046
2F柱(Pu=153.19, Mu=141.95): rho=1.200%, util=0.9200
[PASS] 2F柱確認被強度需求govern(rho>1.0%)

1F樑: 上緣As=11.46cm2(#6(D19)x4), 下緣As=5.73cm2(#6(D19)x2)
屋頂樑: 上緣As=9.93cm2(#5(D16)x5), 下緣As=7.74cm2(#7(D22)x2)


## 第 3 課:正負彎矩要求(`ACI 318-19 18.6.3.1`)

**`1F` 樑比值非常貼近門檻,特別標注,不是輕鬆通過。**

In [5]:

Mn_1F_neg = r_1F_neg['phiMn_provided']/0.9
Mn_1F_pos = r_1F_pos['phiMn_provided']/0.9
Mn_roof_neg = r_roof_neg['phiMn_provided']/0.9
Mn_roof_pos = r_roof_pos['phiMn_provided']/0.9

ratio_1F = Mn_1F_pos/Mn_1F_neg
ratio_roof = Mn_roof_pos/Mn_roof_neg
print(f"1F樑: Mn-={Mn_1F_neg:.2f}, Mn+={Mn_1F_pos:.2f}, ratio={ratio_1F:.3f}(>=0.5): {'PASS' if ratio_1F>=0.5 else 'FAIL'}")
print(f"屋頂樑: Mn-={Mn_roof_neg:.2f}, Mn+={Mn_roof_pos:.2f}, ratio={ratio_roof:.3f}(>=0.5): {'PASS' if ratio_roof>=0.5 else 'FAIL'}")

assert ratio_1F >= 0.5 and ratio_roof >= 0.5
if ratio_1F < 0.55:
    print(f"\n[注意] 1F樑比值{ratio_1F:.3f}非常貼近門檻0.5, 不是輕鬆通過, 值得特別留意")


1F樑: Mn-=190.83, Mn+=99.39, ratio=0.521(>=0.5): PASS
屋頂樑: Mn-=167.20, Mn+=132.41, ratio=0.792(>=0.5): PASS

[注意] 1F樑比值0.521非常貼近門檻0.5, 不是輕鬆通過, 值得特別留意


## 第 4 課:`SCWB`(`ACI 318-19 18.7.3`)

**比值從原本(集中力模型)的 `2.99` 降到這次(真實分佈載重)的更低值
——梁配筋大幅增加,`Mnb` 跟著增加,餘裕被壓縮。**

In [6]:

sum_Mnc_1F = result_1F_col['Mn'] + result_2F_col['Mn']
ratio_scwb_1F = sum_Mnc_1F/Mn_1F_neg
print(f"1F接頭: sum(Mnc)={sum_Mnc_1F:.2f}, Mnb={Mn_1F_neg:.2f}, ratio={ratio_scwb_1F:.2f}(>=1.2): "
      f"{'PASS' if ratio_scwb_1F>=1.2 else 'FAIL'}")
assert ratio_scwb_1F >= 1.2

limit_exempt = (1600.0*280.0/10)*9.80665e-3
roof_exempt = Pu_2F_col < limit_exempt
print(f"\n屋頂接頭: 豁免條件(Pu={Pu_2F_col}<{limit_exempt:.2f}): {roof_exempt}")
print("豁免(ACI 318-19 18.7.3.1, 柱不連續+低軸力), 不需要SCWB檢核" if roof_exempt else "需要檢核")


1F接頭: sum(Mnc)=317.42, Mnb=190.83, ratio=1.66(>=1.2): PASS

屋頂接頭: 豁免條件(Pu=153.19<439.34): True
豁免(ACI 318-19 18.7.3.1, 柱不連續+低軸力), 不需要SCWB檢核


## 第 5 課:`Capacity-Design Shear`(梁 + 柱)

In [7]:

def compute_Mpr(As_cm2, b_cm, d_cm, fc=280.0, fy=4200.0):
    fy_pr = 1.25*fy
    a = As_cm2*fy_pr/(0.85*fc*b_cm)
    return (As_cm2*fy_pr*(d_cm - a/2)) * 9.80665e-5

Mpr_1F_neg = compute_Mpr(r_1F_neg['As_provided'], 30.0, r_1F_neg['d'])
Mpr_1F_pos = compute_Mpr(r_1F_pos['As_provided'], 30.0, r_1F_pos['d'])
Mpr_roof_neg = compute_Mpr(r_roof_neg['As_provided'], 30.0, r_roof_neg['d'])
Mpr_roof_pos = compute_Mpr(r_roof_pos['As_provided'], 30.0, r_roof_pos['d'])

Ln_beam = L_bay - h_col
Ve_hinge_1F_beam = (Mpr_1F_neg + Mpr_1F_pos)/Ln_beam
Ve_hinge_roof_beam = (Mpr_roof_neg + Mpr_roof_pos)/Ln_beam
print(f"1F樑: Mpr(上)={Mpr_1F_neg:.2f}, Mpr(下)={Mpr_1F_pos:.2f}, Ve_hinge={Ve_hinge_1F_beam:.2f}kN")
print(f"屋頂樑: Mpr(上)={Mpr_roof_neg:.2f}, Mpr(下)={Mpr_roof_pos:.2f}, Ve_hinge={Ve_hinge_roof_beam:.2f}kN")

def beam_stirrup_spacing(Ve_hinge, d_beam, db_long=1.9):
    Vn_req = Ve_hinge/0.75
    Vs_req_kgf = Vn_req/9.80665e-3
    s_from_Vs = (2*0.7133*4200.0*d_beam)/Vs_req_kgf
    s_max_seismic = min(d_beam/4, 6*db_long, 15.0)
    return min(s_from_Vs, s_max_seismic)

s_1F_beam = beam_stirrup_spacing(Ve_hinge_1F_beam, r_1F_neg['d'])
s_roof_beam = beam_stirrup_spacing(Ve_hinge_roof_beam, r_roof_neg['d'])
print(f"\n1F樑箍筋間距={s_1F_beam:.2f}cm, 屋頂樑箍筋間距={s_roof_beam:.2f}cm")

# 柱capacity-design shear
fy_pr_col = 1.25*4200.0
for label, Pu, rho_col, Mpr_beam_adj in [
    ('1F柱', Pu_1F_col, result_1F_col['rho'], Mpr_1F_neg),
    ('2F柱', Pu_2F_col, result_2F_col['rho'], Mpr_roof_neg),
]:
    As_bar = rho_col*40.0*40.0/8
    As_layers = [3*As_bar, 2*As_bar, 3*As_bar]
    r_pr = design_column_PM(b_cm=40.0, h_cm=40.0, As_per_layer_cm2=As_layers,
                              cover_to_center_cm=4.0, n_layers=3,
                              fc=280.0, fy=fy_pr_col, Es=2.0e6, Pu_kN=Pu, Mu_kNm=1.0)
    Mpr_col = Mn_at_Pu(r_pr['curve'], Pu)
    Ln_col = 3.0
    Ve_own = 2*Mpr_col/Ln_col
    Ve_beam = Mpr_beam_adj/Ln_col
    Ve_gov = min(Ve_own, Ve_beam)
    print(f"\n{label}: Mpr_own={Mpr_col:.2f}, Ve(own)={Ve_own:.2f}, Ve(beam)={Ve_beam:.2f}, governing={Ve_gov:.2f}kN")


1F樑: Mpr(上)=233.57, Mpr(下)=123.00, Ve_hinge=63.67kN
屋頂樑: Mpr(上)=205.26, Mpr(下)=163.24, Ve_hinge=65.80kN

1F樑箍筋間距=10.95cm, 屋頂樑箍筋間距=10.95cm

1F柱: Mpr_own=184.31, Ve(own)=122.87, Ve(beam)=77.86, governing=77.86kN

2F柱: Mpr_own=187.44, Ve(own)=124.96, Ve(beam)=68.42, governing=68.42kN


## 第 6 課:柱端圍束

In [8]:

def design_column_confinement(h_col_cm, Ln_cm, db_long_cm, fc=280.0, fyt=4200.0, cover_cm=4.0):
    lo = max(h_col_cm, Ln_cm/6, 45.0)
    bc = h_col_cm - 2*cover_cm
    so = min(max(10.0 + (35.0-bc)/3.0, 10.0), 15.0)
    s_max = min(h_col_cm/4, 6*db_long_cm, so)
    Ag = h_col_cm**2
    Ach = bc**2
    Ash_over_s_req = max(0.3*bc*(fc/fyt)*(Ag/Ach-1), 0.09*bc*fc/fyt)
    Ash_2leg = 2*0.7133
    return dict(lo=lo, s_max=s_max, s_final=min(Ash_2leg/Ash_over_s_req, s_max))

conf_1F = design_column_confinement(40.0, (h1-0.5)*100, 1.6)
conf_2F = design_column_confinement(40.0, (h2-0.5)*100, 1.8)
print(f"1F柱圍束: lo={conf_1F['lo']:.1f}cm, s={conf_1F['s_final']:.2f}cm")
print(f"2F柱圍束: lo={conf_2F['lo']:.1f}cm, s={conf_2F['s_final']:.2f}cm")


1F柱圍束: lo=50.0cm, s=3.96cm
2F柱圍束: lo=50.0cm, s=3.96cm


## 第 7 課:梁柱接頭剪力

**`1F` 接頭餘裕從原本 `2.28` 降到接近臨界值,特別標注。**

In [9]:

def joint_shear_check(As_top, Mpr_beam, h_avg):
    T1_kN = (As_top*5250.0)*9.80665e-3
    Vcol_kN = Mpr_beam/h_avg
    Vj_kN = T1_kN - Vcol_kN
    fc_psi = 280.0*14.223
    Aj_in2 = 1600.0/6.4516
    Vn_kN = (10*(fc_psi**0.5)*Aj_in2/1000)*4.448
    phiVn_kN = 0.85*Vn_kN
    return dict(Vj_kN=Vj_kN, phiVn_kN=phiVn_kN, ratio=phiVn_kN/Vj_kN)

j1 = joint_shear_check(r_1F_neg['As_provided'], Mpr_1F_neg, (h1+h2)/2)
j2 = joint_shear_check(r_roof_neg['As_provided'], Mpr_roof_neg, h2)
print(f"1F接頭: Vj={j1['Vj_kN']:.2f}, phiVn={j1['phiVn_kN']:.2f}, ratio={j1['ratio']:.2f}, {'PASS' if j1['ratio']>=1.0 else 'FAIL'}")
print(f"屋頂接頭: Vj={j2['Vj_kN']:.2f}, phiVn={j2['phiVn_kN']:.2f}, ratio={j2['ratio']:.2f}, {'PASS' if j2['ratio']>=1.0 else 'FAIL'}")

assert j1['ratio'] >= 1.0 and j2['ratio'] >= 1.0
if j1['ratio'] < 1.2:
    print(f"\n[注意] 1F接頭餘裕{j1['ratio']:.2f}倍已經很薄(原本集中力模型是2.28倍),")
    print("這是整個重做結果裡最需要留意的一項, 不是輕鬆通過")


1F接頭: Vj=523.28, phiVn=591.71, ratio=1.13, PASS
屋頂接頭: Vj=452.60, phiVn=591.71, ratio=1.31, PASS

[注意] 1F接頭餘裕1.13倍已經很薄(原本集中力模型是2.28倍),
這是整個重做結果裡最需要留意的一項, 不是輕鬆通過


## 小結:一張網路圖揭露的真實建模缺口,讓整個設計全面改觀

- 梁上真實分佈載重 vs 原本的集中力簡化,讓梁端彎矩暴增 `6.7` 倍、
  `2F` 柱彎矩暴增超過 `12` 倍
- `2F` 柱第一次真正被強度需求(不是規範最小值)`govern`
- 所有 `7` 項檢核都重新做完且全部 `PASS`,但 `1F` 梁正負彎矩比值
  (`0.52`附近)跟 `1F` 接頭剪力餘裕都明顯比之前貼近門檻,不是
  輕鬆過關——這才是更真實、更有意義的設計結果
- 過程中真的踩過一次坑(跨中彎矩解析公式符號算錯,差了近 `5` 倍),
  改用切分子元素的方式重新驗證才找到正確答案